In [1]:
from kfunc_filter import should_filter_function
from collections import defaultdict, deque
from pathlib import Path
import networkx as nx
import os
import scipy
import re
import time
import subprocess

start_time = time.perf_counter()
the_directory = 'output/demo'
demo_profile = "./demo-profile-sysclose.txt"
search_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
kernel_cg_path = "./pruned_callgraph.graphml"
k_cg = nx.read_graphml(kernel_cg_path)
k_cg.remove_edges_from(nx.selfloop_edges(k_cg))
kall_functions = [node for node in k_cg.nodes if not should_filter_function(node)]

def gen_subgraph(static_graph, sys_entry_function, function_set, remove_broken_edges=False):
    function_set = [f for f in function_set if not should_filter_function(f)]
    subgraph_nodes = set(function_set)
    if remove_broken_edges:
        reachable_nodes = nx.descendants(static_graph, sys_entry_function)
        reachable_nodes.add(sys_entry_function)
        subgraph_nodes = subgraph_nodes.intersection(reachable_nodes)
    subgraph = static_graph.subgraph(subgraph_nodes).copy()
    return subgraph

def outedges_callgraph(graph, function_name, do_filter=True):
    if function_name not in graph:
        print(f"Function {function_name} does not exist in the graph.")
        return []
    out_edges = []
    for _, target, edge_data in graph.out_edges(function_name, data=True):
        if do_filter and should_filter_function(target):
            continue
        out_edges.append((function_name, target, edge_data))
    return out_edges

def export_graph_as_markdown_tree(graph):
    def dfs(node, indent=0, visited=set()):
        lines = []
        prefix = "  " * indent + "- " + node
        lines.append(prefix)
        visited.add(node)
        for _, neighbor in graph.out_edges(node):
            if neighbor not in visited:
                lines.extend(dfs(neighbor, indent + 1, visited))
        return lines
    roots = [n for n in graph.nodes if graph.in_degree(n) == 0]
    all_lines = []
    for root in roots:
        all_lines.extend(dfs(root, indent=0, visited=set()))
    return "\n".join(all_lines)

def find_function_source_in_file(path: str, function_name: str) -> str | None:
    sig_re = re.compile(rf'\b{re.escape(function_name)}\b\s*\(')
    def_re = re.compile(
        rf'''^[ \t]*
            (?:[A-Za-z_]\w*(?:\s*\*+)?\s+)+
            \**\s*
            {re.escape(function_name)}\s*\(
        ''',
        re.VERBOSE
    )
    try:
        f = open(path, 'r', errors='ignore')
    except PermissionError:
        return None
    with f:
        collecting = False
        brace_count = 0
        buffer = []
        in_block = False
        for raw in f:
            line = raw
            if in_block:
                end = line.find('*/')
                if end >= 0:
                    in_block = False
                    line = line[end+2:]
                else:
                    continue
            start = line.find('/*')
            if start >= 0:
                end = line.find('*/', start+2)
                if end >= 0:
                    line = line[:start] + line[end+2:]
                else:
                    in_block = True
                    line = line[:start]
            clean = line.split('//',1)[0]
            if clean.rstrip().endswith('\\'):
                continue
            if not collecting:
                if sig_re.search(clean) and def_re.match(clean):
                    sig_raw = [raw]
                    sig_clean = [clean]
                    bal = clean.count('(') - clean.count(')')
                    while bal > 0:
                        nxt_raw = f.readline()
                        if not nxt_raw:
                            break
                        nxt = nxt_raw
                        if in_block:
                            end = nxt.find('*/')
                            if end >= 0:
                                in_block = False
                                nxt = nxt[end+2:]
                            else:
                                continue
                        st = nxt.find('/*')
                        if st >= 0:
                            ed = nxt.find('*/', st+2)
                            if ed >= 0:
                                nxt = nxt[:st] + nxt[ed+2:]
                            else:
                                in_block = True
                                nxt = nxt[:st]
                        nxt_clean = nxt.split('//',1)[0]
                        sig_raw.append(nxt_raw)
                        sig_clean.append(nxt_clean)
                        bal += nxt_clean.count('(') - nxt_clean.count(')')
                    if ''.join(sig_clean).strip().endswith(';'):
                        continue
                    buffer = sig_raw.copy()
                    if '{' in sig_clean[-1]:
                        collecting = True
                        brace_count = sig_clean[-1].count('{') - sig_clean[-1].count('}')
                    else:
                        for body_raw in f:
                            body = body_raw
                            if in_block:
                                end = body.find('*/')
                                if end >= 0:
                                    in_block = False
                                    body = body[end+2:]
                                else:
                                    continue
                            st = body.find('/*')
                            if st >= 0:
                                ed = body.find('*/', st+2)
                                if ed >= 0:
                                    body = body[:st] + body[ed+2:]
                                else:
                                    in_block = True
                                    body = body[:st]
                            body_clean = body.split('//',1)[0]
                            buffer.append(body_raw)
                            if '{' in body_clean:
                                collecting = True
                                brace_count = (
                                    body_clean.count('{') -
                                    body_clean.count('}')
                                )
                                break
                    if not collecting:
                        buffer = []
            else:
                buffer.append(raw)
                tmp = raw
                if in_block:
                    end = tmp.find('*/')
                    if end >= 0:
                        in_block = False
                        tmp = tmp[end+2:]
                    else:
                        continue
                st = tmp.find('/*')
                if st >= 0:
                    ed = tmp.find('*/', st+2)
                    if ed >= 0:
                        tmp = tmp[:st] + tmp[ed+2:]
                    else:
                        in_block = True
                        tmp = tmp[:st]
                tmp_clean = tmp.split('//',1)[0]
                brace_count += tmp_clean.count('{') - tmp_clean.count('}')
                if brace_count == 0:
                    return ''.join(buffer)
    return None

def find_function_native(root_dir: str, function_name: str) -> str | None:
    pat = rf'{re.escape(function_name)}[[:space:]]*\('
    try:
        out = subprocess.check_output([
            'grep', '-RlE',
            '--include=*.c', pat, root_dir
        ], text=True, stderr=subprocess.DEVNULL)
    except subprocess.CalledProcessError:
        return None
    for file_path in out.splitlines():
        src = find_function_source_in_file(file_path, function_name)
        if src:
            return src
    return None

demo_profiled_functions = set()

with open(demo_profile, "r") as f:
    lines = f.readlines()
    for line in lines:
        func_name = line.strip()
        if func_name in kall_functions and not should_filter_function(func_name):
            demo_profiled_functions.add(func_name)

subgraph = gen_subgraph(k_cg, "__x64_sys_close", demo_profiled_functions)
subgraph_start = subgraph.copy()

print('Done!')

Done!


In [2]:
is_there_something_new = True
has_been_checked = []

while (is_there_something_new):
    is_there_something_new = False
    topo_sorted = list(nx.topological_sort(subgraph_start))
    reverse_topo_sorted = list(reversed(topo_sorted))
    try_this = defaultdict(list)
    for i in reverse_topo_sorted:
        out_edges = outedges_callgraph(k_cg, i)
        for src, dst, edge_data in out_edges:
            try_this[src].append(dst)
    markdown_output = export_graph_as_markdown_tree(subgraph_start)
    for src, targets in try_this.items():
        found_function_code = 'Some function code.'
        for dst in targets:
            if subgraph_start.has_edge(src, dst):
                continue
            if dst in has_been_checked:
                continue
            else:
                has_been_checked.append(dst)
            is_there_something_new = True
            subgraph_start.add_edge(src, dst, inferred=True)

markdown_output = export_graph_as_markdown_tree(subgraph_start)
the_markdown = f'{the_directory}/the_start_markdown.txt'

with open(the_markdown, 'w', encoding='utf-8') as file:
    file.write(markdown_output)

print('Done!')

Done!


In [3]:
the_dictionary_functions = {}
file_path = f"{the_directory}/the_functions_all.txt"
ending = 'XXXTHISENDSHEREXXX'

with open(file_path, "w") as f:
    f.write("")

with open(file_path, "a") as f:
    for node in subgraph_start.nodes():
        src = find_function_native(search_dir, node)
        if src is not None:
            the_dictionary_functions[node] = src
            f.write(f"Source Code for {node}:\n")
            f.write(src)
            f.write(f"{ending}\n\n")

print('Done!')

In [4]:
end_time = time.perf_counter()
elapsed = end_time - start_time
hours   = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = elapsed % 60

print(f"Elapsed time: {hours}h {minutes}m {seconds:.6f}s")
print('Done!')

Elapsed time: 0h 20m 27.461172s
Done!
